# REBYU exam-question generator: LoRA fine-tune

Fine-tunes an instruct model to write **original** certification exam questions
(MCQ, short answer, descriptive, programming, UML diagram) in the style of
IT Passport, FE and TOPCIT.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier works; an L4 or A100 is ~3-5× faster).

**Before you start**, put these four files in a Google Drive folder, `MyDrive/rebyu_finetune/` by default:

| File | From |
|---|---|
| `train.jsonl`, `val.jsonl`, `reference.jsonl` | `python-backend/scripts/finetune_dataset/out/` |
| `colab_mix.py`, `check_copy.py` | `python-backend/scripts/finetune_dataset/` |

⚠️ The data contains ITPEC/IPA past-paper text. Keep this notebook and the Drive folder **private**, and never `push_to_hub` the dataset or a public model trained on it.

## 1. Install

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade datasets

## 2. Settings
Everything you are likely to change is here.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/rebyu_finetune"

# 7B is the quality pick; on a free T4 one epoch takes a few hours, and
# checkpoints are saved to Drive so a disconnect only costs the last stretch.
# For a quick first run use "unsloth/Qwen2.5-3B-Instruct-bnb-4bit".
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
CHAT_TEMPLATE = "qwen-2.5"

MAX_SEQ_LEN = 2048
LORA_R = 16
EPOCHS = 1
LR = 2e-4
BATCH = 2
GRAD_ACCUM = 4          # effective batch 8
MAX_STEPS = -1          # e.g. 60 for a smoke test; -1 = full EPOCHS
USE_PUBLIC_MIX = True   # False = train on the REBYU bank only

OUTPUT_DIR = DRIVE_DIR + "/checkpoints"
ADAPTER_DIR = DRIVE_DIR + "/rebyu-qgen-lora"

## 3. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.chdir(DRIVE_DIR)
for f in ["train.jsonl", "val.jsonl", "reference.jsonl", "colab_mix.py", "check_copy.py"]:
    print(("ok      " if os.path.exists(f) else "MISSING ") + f)

## 4. Build the mixed dataset
Mixes the bank with OpenOrca, verifiable-coding-problems and the UML generator set. The coding set needs a Hugging Face login (free account, read token).

**Check the printed counts:** a `0` for code or uml means that dataset's column names changed. Look at the printed `columns:` line and adjust `pick()` / the field names in `colab_mix.py`.

In [ ]:
if USE_PUBLIC_MIX:
    # Add the token in the sidebar's Secrets panel (key icon) as HF_TOKEN with
    # notebook access on. It goes into the environment so the mix script,
    # which runs as a separate process, can read it.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
if USE_PUBLIC_MIX:
    # Runs as a separate process so every byte it used is freed before the
    # model loads; inside the notebook kernel it stayed resident and the
    # session ran out of RAM.
    # Always rebuild: a half-written file left by a crashed run would otherwise
    # be picked up silently.
    !rm -f mixed_train.jsonl mixed_val.jsonl
    !python colab_mix.py
    # A failing `!` command does not stop the notebook, so check explicitly.
    if not os.path.exists("mixed_train.jsonl"):
        raise RuntimeError("colab_mix.py failed. Read the error above, or set USE_PUBLIC_MIX = False.")
    TRAIN_FILE, VAL_FILE = "mixed_train.jsonl", "mixed_val.jsonl"
else:
    TRAIN_FILE, VAL_FILE = "train.jsonl", "val.jsonl"
print(TRAIN_FILE, VAL_FILE)

## 5. Load the model and attach LoRA

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,                         # auto: bf16 on Ampere+, fp16 on T4
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=14,
)

## 6. Format the data
Applies the chat template, then drops rows longer than `MAX_SEQ_LEN`. Truncating them would cut the JSON answer mid-way and teach the model to emit broken JSON.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
ds = ds.remove_columns([c for c in ds["train"].column_names if c != "messages"])

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}

def fits(batch):
    return [len(tokenizer(t, add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LEN for t in batch["text"]]

ds = ds.map(to_text, batched=True, remove_columns=["messages"])
before = {k: len(v) for k, v in ds.items()}
ds = ds.filter(fits, batched=True)
print("rows before:", before, "after length filter:", {k: len(v) for k, v in ds.items()})

# Keep evaluation quick: a fixed 300-row slice of validation is plenty to watch the loss.
eval_ds = ds["validation"].shuffle(seed=14).select(range(min(300, len(ds["validation"]))))
print(ds["train"][0]["text"][:1500])

## 7. Train
Loss is computed on the **assistant** turn only: the model is trained to write the question, not to repeat the prompt and style references back.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        output_dir=OUTPUT_DIR,
        report_to="none",
        seed=14,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Resumes automatically from the latest Drive checkpoint after a disconnect.
import glob
resume = bool(glob.glob(OUTPUT_DIR + "/checkpoint-*"))
print("resuming" if resume else "starting fresh")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

## 8. Save the adapter

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved LoRA adapter to", ADAPTER_DIR)

## 9. Try it
`generate()` builds the same prompt shape the model was trained on, and `validate()` checks that the reply is well-formed for its type.

In [ ]:
import json, random, re

FastLanguageModel.for_inference(model)
SYSTEM = json.loads(open("train.jsonl", encoding="utf-8").readline())["messages"][0]["content"]

TYPE_LABEL = {
    "MCQ": "multiple choice (exactly one correct choice)",
    "SHORT_ANSWER": "short answer (a word or short phrase, auto-graded by exact match)",
    "DESCRIPTIVE": "descriptive / essay (graded against a model answer and rubric)",
    "CRITICAL_THINKING": "programming task (graded by test cases)",
    "DIAGRAM": "diagram (candidate draws a class diagram, graded against a reference)",
}

def generate(cert, topic, qtype="MCQ", difficulty="AVERAGE", references=(), temperature=0.8):
    user = [f"Certification: {cert}", f"Topic: {topic}", f"Question type: {TYPE_LABEL[qtype]}",
            f"Difficulty: {difficulty}"]
    if qtype == "MCQ":
        user += ["Number of choices: 4", "Include explanation: yes"]
    if qtype == "CRITICAL_THINKING":
        user.append("Language: python")
    if references:
        user.append("\nStyle references (do NOT copy):")
        user += [f"--- Reference {i} ---\n{r}" for i, r in enumerate(references, 1)]
    user.append("\nWrite a new question.")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n".join(user)}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=900, temperature=temperature, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def validate(raw):
    try:
        q = json.loads(re.sub(r"^```(?:json)?|```$", "", raw.strip()).strip())
    except ValueError:
        return None, "not JSON"
    t = q.get("type")
    if not q.get("question"):
        return None, "no question"
    if t == "MCQ" and (len(q.get("choices", {})) < 2 or q.get("answer") not in q.get("choices", {})):
        return None, "answer not among choices"
    if t == "SHORT_ANSWER" and not q.get("answer"):
        return None, "no answer"
    if t == "DESCRIPTIVE" and not q.get("model_answer"):
        return None, "no model answer"
    if t == "CRITICAL_THINKING" and not q.get("test_cases"):
        return None, "no test cases"
    if t == "DIAGRAM" and not q.get("reference_diagram"):
        return None, "no reference diagram"
    return q, "ok"

for cert, topic, qt in [
    ("FE Exam", "Technology Element > Database > Database Design", "MCQ"),
    ("IT Passport Exam", "Strategy > Corporate and Legal Affairs > Intellectual Property", "MCQ"),
    ("TOPCIT", "Software Development > Requirements > Requirements Analysis", "SHORT_ANSWER"),
    ("TOPCIT", "Information Security > Security Fundamentals > Cryptography", "DESCRIPTIVE"),
    ("TOPCIT", "Software Development > Programming > Algorithms", "CRITICAL_THINKING"),
    ("Software design (general practice)", "UML modelling", "DIAGRAM"),
]:
    raw = generate(cert, topic, qt)
    q, status = validate(raw)
    print(f"===== {cert} | {qt} | {status}")
    print(json.dumps(q, indent=2, ensure_ascii=False)[:1500] if q else raw[:800])

## 10. Generate a batch and check for copying
The model trained on past papers, so it *can* reproduce one. `check_copy.py` flags anything with 30%+ word-5-gram overlap with a source question. **Reject flagged questions; don't publish them.**

In [ ]:
N = 40
topics = []
for line in open(VAL_FILE if os.path.exists(VAL_FILE) else "val.jsonl", encoding="utf-8"):
    r = json.loads(line)
    meta = r.get("meta") or {}
    if meta.get("cert") and meta.get("topic") and meta.get("type") in TYPE_LABEL:
        topics.append((meta["cert"], meta["topic"], meta["type"]))
random.seed(1)
ok = bad = 0
with open("generated.jsonl", "w", encoding="utf-8") as f:
    for cert, topic, qt in random.sample(topics, min(N, len(topics))):
        q, status = validate(generate(cert, topic, qt))
        if q:
            ok += 1
            f.write(json.dumps(q, ensure_ascii=False) + "\n")
        else:
            bad += 1
print(f"valid: {ok}  invalid: {bad}")

In [ ]:
!python check_copy.py generated.jsonl reference.jsonl 0.30

## 11. Export (optional)
- **Merged 16-bit:** to serve with vLLM / TGI or load with plain `transformers`.
- **GGUF q4_k_m:** to run locally with Ollama or llama.cpp (`ollama create rebyu-qgen -f Modelfile`).

Both write to Drive. Keep the repo private if you upload anywhere.

In [ ]:
EXPORT_MERGED = False
EXPORT_GGUF = False

if EXPORT_MERGED:
    model.save_pretrained_merged(DRIVE_DIR + "/rebyu-qgen-merged", tokenizer, save_method="merged_16bit")
if EXPORT_GGUF:
    model.save_pretrained_gguf(DRIVE_DIR + "/rebyu-qgen-gguf", tokenizer, quantization_method="q4_k_m")

## Reload later without retraining

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
# from unsloth.chat_templates import get_chat_template
# tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)
# FastLanguageModel.for_inference(model)